<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Rapid Application Development<br>using Large Language Models</b></h1>
<h2><b>Notebook 7:</b> Towards Orchestration and Agentics</h2>
<br>

Welcome back to the course environment! In the previous notebook, we ventured into the realm of LLM servers, deploying models capable of handling complex instructions and engaging in conversations. We learned how to interact with these models using various interfaces and touched upon the initial steps of LLM orchestration.

Now, we're going to dive deeper into orchestrating LLMs to build sophisticated applications. Think of LLM orchestration as conducting a symphony, where different components (prompts, retrieval mechanisms, routing strategies, and tools) all play harmoniously to produce powerful end-user applications.

#### **Learning Objectives:**

By the end of this notebook, you will:
- Understand how to approach non-trivial LLM tasks like long-form reasoning and generation.
- Learn retrieval techniques to enhance context and improve LLM outputs.
- Explore routing strategies to direct tasks to appropriate models or tools.
- Discover how to integrate external tools to extend the capabilities of LLMs.
- Develop agentic systems that can reason and act iteratively, culminating in the ReAct loop.


In [1]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

model_path = "http://127.0.0.1:9004/v1"
%env NVIDIA_BASE_URL=$model_path
%env NVIDIA_DEFAULT_MODE=open

model_name = "nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16"

llm = ChatNVIDIA(
    model=model_name, 
    base_url=model_path, 
    max_completion_tokens=4000, 
    temperature=0
)

llm.invoke("Hello World")

env: NVIDIA_BASE_URL=http://127.0.0.1:9004/v1
env: NVIDIA_DEFAULT_MODE=open


AIMessage(content='Hello! How can I assist you today?\n', additional_kwargs={}, response_metadata={'role': 'assistant', 'content': 'Hello! How can I assist you today?\n', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning_content': None, 'token_usage': {'prompt_tokens': 18, 'total_tokens': 28, 'completion_tokens': 10, 'prompt_tokens_details': None}, 'finish_reason': 'stop', 'model_name': 'nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16'}, id='run--00c084a8-7b09-4787-916c-856567d03f19-0', usage_metadata={'input_tokens': 18, 'output_tokens': 10, 'total_tokens': 28}, role='assistant')

<hr>
<br>

## **Part 7.1:** Structuring LLM Workflows

Prompt engineering is the art and science of crafting inputs that guide LLMs to produce desired outputs. Since the models are inherently **stochastic parrots** at heart - meaning they output probabilistic responses based on their input and training data - you can expect certain tasks to be far easier than others for even the most powerful models. Fortunately, we know exactly what most of these models are best at: **"summarization" or synthesis-style tasks**.

**The reasons for a specialty in short-form generation is actually quite simple:**
- Long-form text generation is harder to keep on track and LLMs are likely to derail due to accumulating error of autoregressive sampling.
- Most developers want to conserve generation length and would rather have accidentally-overshort responses over accidentally-overlong ones.
- In chat applications (which are usually the most popular default models, and the most popular instruction format to work with), shorter-form, concise, and "summary-like" responses are often preferred in real-world scenarios.

**With that being said, long context reasoning is valued greatly:**
- The ability to specify long outputs to help reinforce generation style/format is highly desirable from an end-user perspective.
- Even if the generative prior for responses is towards short outputs, the accumulation of said short outputs can become very long very quickly.

**For these reasons, most models tend to be trained for short-form generation and long-form context ingestion.** This makes LLMs perfect for tasks like summarization and long-form question-answering, which at its core boils down to the problem of **knowledge synthesis** or **distillation**. This idea was already explored in code in the previous notebook, but it's important to formalize and keep in mind going forward.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import json
import os 

###################################################################################################
## Context-Fetching Utilities

def notebook_to_markdown(path: str) -> str:
    """Load a Jupyter notebook from a given path and convert it to Markdown format."""
    with open(path, 'r', encoding='utf-8') as file:
        notebook = json.load(file)
    markdown_content = []
    for cell in notebook['cells']:
        if cell['cell_type'] == 'code':          # Combine code into one block
            markdown_content += [f'```python\n{"".join(cell["source"])}\n```']
        elif cell['cell_type'] == 'markdown':    # Directly append markdown source
            markdown_content += ["".join(cell["source"])]
    return '\n\n'.join(markdown_content)

def compute_context(state: dict):
    full_context = [f"\n[[[{file}]]]\n\n{notebook_to_markdown(file)}" for file in state.get("filenames")]
    return "\n\n".join(full_context)

filenames = [v for v in sorted(os.listdir(".")) if v.endswith(".ipynb")]

###################################################################################################
## Prompt/Pipeline Definition

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful DLI Chatbot who can request and reason about notebooks."
        " Be as concise as necessary, but follow directions as best as you can."
        " Please help the user out by answering any of their questions and following their instructions."
    )),
    ("human", "Here is the notebook I want you to work with: {full_context}. Remembering this, start the conversation over."),
    ("ai", "Awesome! I will work with this as context and will restart the conversation."),
    ("placeholder", "{messages}")
])

pipeline = (
    RunnablePassthrough.assign(full_context = compute_context)
    | chat_prompt 
    | llm   ## requires shorter context
    # | llm.bind(model="mistralai/mixtral-8x22b-instruct-v0.1")
    # | llm.bind(model="nvidia/nemotron-3.5-lightning-30b-a3b")
    | StrOutputParser()
)

chat_state = {
    ## Reason just about the previous notebook
    "filenames": [next(name for name in filenames if name.startswith("01_llm_intro"))],
    "messages": [("human", "Can you give me a summary of the notebook?")],
}

short_summary = ""
for chunk in pipeline.stream(chat_state):
    print(chunk, end="")
    short_summary += chunk

**Summary of Notebook 1: Getting Started With Large Language Models**

This notebook introduces the foundational concepts and practical steps for working with Large Language Models (LLMs) using HuggingFace tools. Key points include:

1. **Deep Learning Recap**:  
   - Progresses from basic linear models to advanced architectures (e.g., transformers), emphasizing pre-trained models like BERT for handling high-dimensional data.

2. **LLM Introduction**:  
   - Highlights HuggingFace as a primary tool for accessing pre-trained models (e.g., `bert-base-uncased`).  
   - Demonstrates a "fill-mask" pipeline to showcase LLM capabilities with minimal code.

3. **Pipeline Dissection**:  
   - Breaks down the pipeline into three stages:  
     - **Preprocess**: Tokenizer converts text to tensors.  
     - **Forward**: Model processes tensors into outputs.  
     - **Postprocess**: Converts outputs back to human-readable results.  
   - Provides a custom pipeline class to illustrate internal work

<br>

### Breaking Free From Limitations

Despite the natural propensities of modern models, people are still able to do complex tasks like long-form generation and even longer-form context ingestion through a variety of scalable ways. **These techniques rely on the key assumption that an LLM can be called programmatically, repeatedly, independently, and concurrently (all of which a server-based deployment allows).**

<div><img src="imgs/data-pipelines.png" width="800"/></div>

#### **Iterative Generation** 

Much like the way that dialog is technically a long-form progressive generation task (in which half of the generation is done by a human), an arbitrarily long-form document can be generated one piece at a time. If an LLM is trained to only output 5000 tokens at a time - but can reason about 100,000 tokens at a time, then perhaps you could generate a full-length document one step at a time?
- **Full-Context:** In the case that iterative generation stays within the input context limits of the model, then the full-context approach of just keeping the conversational history, task history, document history, etc. may be sufficient. 
- **Running State:** In the case when long context accumulation is undesirable, a running summary or other type of accumulated history from previous generations may be sufficient. This is often called **"iterative refinement"** and has the advertised benefit of maintaining **global context** (or at least hindsight context, with global obtainable through pre-processing). 
- **Sliding Window:** When you want to refine a document and only care about **local context** (i.e. for boundary consistency), you may elect to discard the overall history and only consider a window of content at a time. This is especially relevant for **translation tasks** where you can translate a document slice while remembering both the previously-translated chunk and the subsequent yet-untranslated chunk. 

In [3]:
message_prompt = (
    "Give me a structured summary of the course, making sure to note all sections and points?"
    " This is a one-notebook-at-a-time process, and the next notebook is the one provided in context."
    "\n\nThe following is a running summary of previous notebooks: \n\n{running_summary}\n\n"
    "Output only the summary of the currently-provided notebook, but explain the logical connections to the rest of the course."
    "Make sure the descriptions are also dense in key words that would be useful for searching through the notebooks via a bibliography."
    " Only output the following format, with no other structures, extra newlines, or info not grounded in context:\n<notebook>.ipynb"
    "\n - <Section 1 Name (As Seen In Notebook)>: Decent Description, including frameworks used, important topics, etc."
    "\n - ..."
    "\n - Main Ideas and Relevance To Course: Decent Description, dense with key features/frameworks/topics for bibliographic/semantic lookup."
    "\n - Important Code: Types of syntaxes, variables, etc that are more specific to this notebook, like classes, variables, topics, terms, etc."
    "\n - Connections to previous notebooks: Decent Description, dense with key features/frameworks/topics for bibliographic/semantic lookup."
    "\n - Relevant Images: Brief descriptions of images (i.e. <img src='imgs/url.png'>, no non-image files) with local-scope URLs."
)

running_summaries = []
running_summary = "No summary yes. This is the first notebook."

pipeline = (
    RunnablePassthrough.assign(full_context = compute_context)
    | chat_prompt 
    # | llm
    # | llm.bind(model="mistralai/mixtral-8x22b-instruct-v0.1")
    | llm.bind(model="nvidia/nemotron-3.5-lightning-30b-a3b")
    | StrOutputParser()
)

for name in filenames:
    if not name.endswith(".ipynb"):
        continue
    buffer = ""
    chat_state = {
        "filenames": [name],
        # "messages": [("human", message_prompt.format(running_summary=running_summary))]                    ## Full history
        "messages": [("human", message_prompt.format(running_summary=(running_summaries or ["None"])[-1]))]  ## Sliding window
        # "messages": [("human", message_prompt.format(running_summary="Not provided"))]                     ## No history, 
    }
    for chunk in pipeline.stream(chat_state):
        buffer += chunk
        print(chunk, end="")
    print("\n" + "*" * 84)
    running_summaries += [buffer]
    running_summary = "\n\n".join(running_summaries)
    # break  ## If commented out, causes the notebook to iterate over the entire list

[[[00_jupyterlab.ipynb]]]

 - JupyterLab Interface Overview: Introduction to the JupyterLab dashboard structure including menu bar, file browser sidebar, and main work area; explanation of interactive iPython notebooks, tabbed file views, and launcher page navigation; GPU memory management strategies including soft reset via kernel button, hard stop via kernel menu, and code-based kernel shutdown; practical execution of print statements and code cells using Shift+Enter.

 - Main Ideas and Relevance To Course: Foundational JupyterLab environment setup and navigation; GPU lifecycle management (soft/hard reset mechanisms); interactive code execution workflows; kernel management patterns; establishes the development environment framework for subsequent LLM application notebooks; key terms: JupyterLab, iPython, kernel, GPU memory management, soft reset, hard stop.

 - Important Code: `from composer.course_language import show_language_selector`; `show_language_selector()`; `print('This is j

<details>
<summary><b>Notes On Iterative Refinement:</b></summary>
<ul>
    <li>The example above shows several different strategies (full-context, sliding-window, and no history). You will see that the first two generate different main idea summaries but should be largely the same in this context (but full-context will start to fall about for longer-form refinement problems). No-history will also have some problems (partially because more info is needed and partially because the prompt instructions imply that every notebook is a first notebook).</li>
    <li>Note how dialogue and chain-of-thought (think step-by-step) reasoning are also inherently refinement problems where a response is refined from the full-context (e.g. chat history) which then serves as a new addition for the next refinement step.</li>
    <li>Remember that LLMs strongly benefit from having input that "could have practically been generated". A common cause for hallucination involves bombarding the LLM with context that it should not have been able to generate without adequately informing the LLM that it did not generate it.</li>
</ul>
</details>

**NOTE:** For the chatbot, we can go ahead and summarize all of these into a short-summary using the line below:

In [4]:
pipeline = (
    RunnablePassthrough.assign(full_context = lambda state: running_summary)
    | chat_prompt 
    # | llm
    # | llm.bind(model="mistralai/mixtral-8x22b-instruct-v0.1")
    | llm.bind(model="nvidia/nemotron-3.5-lightning-30b-a3b")
    | StrOutputParser()
)

chat_state = {
    "messages": [("human", 
        "Can you give me a summary of the course, making sure to mention every notebook?"
        " Please start out with a high-level overview, and do not be overly sales-pitchy."
        " Please be compact (2-3 paragraphs), clearly define the material/scope, and finish by"
        " explaining big-picture ideas to help an instructor explain the material and understand"
        " which parts of the course to refer to when addressing questions."
    )],
}

short_summary = ""
for chunk in pipeline.stream(chat_state):
    print(chunk, end="")
    short_summary += chunk

This course, "Rapid Application Development using Large Language Models," provides a structured progression from JupyterLab environment setup through advanced LLM deployment and agentic workflows. It begins with [[[00_jupyterlab.ipynb]]], establishing the foundational dashboard navigation, kernel management (soft reset/hard stop), and execution patterns that underpin all subsequent notebooks. The core LLM content then unfolds across [[[01_llm_intro.ipynb]]] through [[[05_multimodal.ipynb]]], covering deep learning recaps, HuggingFace model loading via pipelines, tokenizer/embedding mechanics, classification heads (token, sequence, zero-shot), and encoder-decoder/decoder-only architectures for translation and generation. Multimodal encoding extends these concepts to audio, vision, and diffusion models. Infrastructure and deployment follow with [[[065_running_server.ipynb]]] and [[[06_llm_server.ipynb]]], which detail FastAPI/vLLM server construction, OpenAI-compatible endpoints, concurr

<br>

#### **Parallel Generation**

In some contexts, iterative generation may be unnecessarily slow or may accumulate an intractable amount of context or error. In these cases, you may want to independently reason about a bunch of smaller pieces of your input and then join the results together at a later point. 
- **Canonicalization:** If you have a task with largely-independent components, you can reason about them independently and progress them into a standardized representation which can be easier to work with later. For example, making parts of a document shorter, extracting important details, or even expanding an short abstract description to a fuller form.
- **Structurization:** Given the task of extracting global context from a long-form document (AKA summarizing), you could make summaries of a window, then summarize those summaries, and so on and so forth until you have a singular summary. Similar ideas of structurization including [**insertion into knowledge graphs with value-edge-value**](https://neo4j.com/developer-blog/knowledge-graph-llama-nvidia-langchain/), [**insertion of embeddings into vector stores for semantic retrieval**](https://arxiv.org/abs/2312.10997), [**and insertion into SQL databases**](https://developer.nvidia.com/blog/new-llm-snowflake-arctic-model-for-sql-and-code-generation/) can happen in a large-scale application of LLMs on large repositories of data.
- **Ensembling:** Given an objective to do something with an LLM, an ensemble of different approaches can be used to independently try to solve a natural-language issue. The result of all of these apporaches can be combined to form the final reasoning.

In [5]:
from langchain_core.runnables import RunnableLambda, RunnableBranch
from functools import partial

message_prompt = (
    " Give me a rigorous summary of only bullet #{section_i} of the outline. Do not summarize any other sections."
    " Output should be a few compact-but-dense paragraphs long and only summarize a fraction of the notebook (bullet {section_i})"
    " such that a reasonable person would be able to understand everything from that section"
    " (while knowing roughly how it ties in with the whole notebook) from the summary." 
)

def llm_route(s):
    if len(s) < 4000:
        return llm
    print("Context too long; let's query server")
    return llm.bind(model="nvidia/nemotron-3.5-lightning-30b-a3b")

pipeline = (
    RunnablePassthrough.assign(full_context = compute_context)
    | chat_prompt
    | RunnableBranch(
        ((lambda x: sum(len(msg.content) for msg in x.messages) > 3000), 
             llm.bind(model="nvidia/nemotron-3.5-lightning-30b-a3b")),
        llm
    ) 
    | StrOutputParser()
)

notebook_chunks = []

def summarize_section(state):
    name, i = state.get("name"), state.get("i")
    print(f"(+{name[1]}.{i})", end="")
    output = pipeline.invoke({
        "filenames": [name],
        "messages": [
            ("human", "Please give me a structured outline of the notebook"), 
            ("ai", state.get("outline")),
            ("human", message_prompt.format(section_i=i))],
    })
    print(f"(-{name[1]}.{i})", end="", flush=True)
    return output

task_keys = []
task_args = []
task_vals = []
num_tasks = 6

for name, outline in zip(filenames, running_summaries):
    task_keys += (nb_keys := [f"Notebook {name} Part {i}" for i in range(1, num_tasks+1)])
    task_args += (nb_args := [{"name": name, "i": i, "outline": outline} for i in range(1, num_tasks+1)])
    ## Notebook-level parallelization: Bottlenecks down to one notebook at a time
    # task_vals += RunnableLambda(summarize_section).batch(nb_args)
    
    # break   ## If commented out, will cause the notebook to iterate over the entire list. 
            ## This section is just for show and is expensive, so no need to run this for all notebooks.
            ## If you do decide to run it, please use the local LLM.

## All-at-once parallelization: Bottleneck is the set maximum concurrency (since threads are hardware-limited)
task_vals = RunnableLambda(summarize_section).batch(task_args)

(+0.1)(+0.2)(+0.3)(+0.4)(+0.5)(+0.6)(+1.1)(+1.2)(+1.3)(+1.4)(+1.5)(+1.6)(+2.1)(+2.2)(+2.3)(+2.4)(+2.5)(+2.6)(+3.1)(+3.2)(+3.3)(+3.4)(+3.5)(+3.6)(+4.1)(+4.2)(+4.3)(+4.4)(-0.5)(+4.5)(-1.4)(+4.6)(-0.3)(+5.1)(-1.3)(+5.2)(-0.1)(+5.3)(-3.2)(+5.4)(-2.5)(+5.5)(-3.1)(+5.6)(-0.2)(+6.1)(-0.6)(+6.2)(-2.6)(+6.3)(-3.3)(+6.4)(-3.4)(+6.5)(-3.5)(+6.6)(-3.6)(+6.1)(-1.6)(+6.2)(-1.5)(+6.3)(-1.1)(+6.4)(-4.2)(+6.5)(-0.4)(+6.6)(-4.3)(+7.1)(-2.1)(+7.2)(-2.3)(+7.3)(-2.2)(+7.4)(-2.4)(+7.5)(-4.5)(+7.6)(-4.1)(+7.1)(-5.1)(+7.2)(-4.4)(+7.3)(-5.2)(+7.4)(-4.6)(+7.5)(-5.3)(+7.6)(-5.5)(+7.1)(-5.4)(+7.2)(-5.6)(+7.3)(-6.3)(+7.4)(-6.2)(+7.5)(-6.5)(+7.6)(-6.4)(+8.1)(-6.2)(+8.2)(-6.6)(+8.3)(-6.5)(+8.4)(-6.1)(+8.5)(-6.3)(+8.6)(-6.6)(+9.1)(-7.2)(+9.2)(-7.1)(+9.3)(-7.3)(+9.4)(-6.1)(+9.5)(-1.2)(+9.6)(-6.4)(+a.1)(-7.6)(+a.2)(-7.3)(+a.3)(-7.4)(+a.4)(-7.5)(+a.5)(-7.2)(+a.6)(-7.1)(-7.4)(-7.2)(-7.5)(-7.6)(-8.1)(-7.4)(-8.2)(-8.4)(-8.6)(-8.5)(-7.6)(-8.3)(-9.3)(-9.2)(-9.4)(-9.6)(-a.1)(-a.3)(-9.5)(-a.4)(-a.2)(-a.6)(-7.1)(-a.5)

Exception: [###] {'message': 'Provider request failed', 'type': 'upstream_error'}
{'error': {'message': 'Provider request failed', 'type': 'upstream_error'}, 'dli_routing': {'reason': 'ttft_timeout', 'model': 'nvidia/nemotron-3.5-lightning-30b-a3b', 'effective_route': 'fallback', 'primary': {'state': 'open', 'consecutive_failures': 6, 'tier': 4, 'retry_after_seconds': 224.805, 'probe_in_flight': False, 'last_failure': 'ttft_timeout', 'attempts': 6, 'successes': 0, 'failed_requests': 6, 'available': False, 'model_allowed': True}, 'proxy_instance_id': 'fbef2ed29247481998ac4600498d709b', 'course_cache_sha256': 'a266e086ee562154bd6c74bb6c609c3420f84ee51a80c7d2a0ae4815abf957f1', 'fallback': {'state': 'closed', 'consecutive_failures': 1, 'tier': 0, 'retry_after_seconds': 0, 'probe_in_flight': False, 'last_failure': 'ttft_timeout', 'attempts': 99, 'successes': 69, 'failed_requests': 1, 'available': True, 'model_allowed': True}, 'policy': {'failure_threshold': 3, 'base_cooldown_seconds': 30.0,

In [6]:
task_vals[0]
task_vals[-1]

IndexError: list index out of range

<details>
<summary><b>Notes On Parallelization:</b></summary>

- If you want to parallelize across inputs to a pipeline, `batch` is a great option that already incorporates thread-safety and concurrency limits (you can guess at maximum concurrency by seeing when the first thread finish occurs). If you wanted to parallelize across different chains (i.e. you want the same data to go through multiple pipes concurrently), you could use `RunnableParallel`. Both are far easier (but also less customizable) than semaphored or thread-pooled approachs which allow much more custom approaches.

- We tweaked our prompt and numbers to try to get a summary for each section, but note how the "summarize part i" objective is ill-defined when the notebook doesn't have enough parts. This is sufficient for our purposes in the assessment, but is definitely not ideal in the grand scheme of things. You will learn strategies later which will mitigate this "language-in -> language-out" problem (***HINT***: see **structured output**).

</details>

**Saving The Results**

To help visualize these results, feel free to run the following cell to save them to a JSON file. These exports - when applied on the whole document pool - can be used to construct a chatbot interface similar to the one accessible to you (see [Table of Contents Notebook](./Table_of_Contents.ipynb)), so it's interesting to see what came out of this pipeline. 

In [ ]:
nbsummary = {
    "course": "NVIDIA Deep Learning Institute's Instructor-Led Course called \"Rapid Application Development with Large Language Models\"",
    "summary": short_summary, 
    "filenames": filenames
}

for i, (name, outline) in enumerate(zip(filenames, running_summaries)):
    if not name.endswith(".ipynb"): continue
    nbsummary[name] = {"outline": outline}
    task_iter = range(i*num_tasks, (i+1)*num_tasks)
    nbsummary[name]["sections"] = [task_vals[j] for j in task_iter]
        
import json
json.dump(nbsummary, open('notebook_chunks.json', "w"), sort_keys=True, indent=4)

# with open('notebook_chunks.json', 'r') as fp:
#     nbsummary = json.load(fp)

#### **Progressing Beyond Model Priors**

These formulations allow the large-language model to not only reason with but also create arbitrarily long and arbitrarily short responses. When implemented programmatically, they also give you control over context scope and bottlenecks, allowing for a trade-off between lightweight limited-scope operations and well-informed slow processes. 

However, these techniques alone are just engineerings on top of a language model, and are self-contained to the capabilities and knowledge base of its immediate context, its prompts, and its priors. **The biggest potential of LLMs lies not in what's static inside the model weights and a pre-computed workflow, but rather its incorporation with a rich environment to guide its progression and accept its responses as actions.**

<hr>
<br>

## **Part 7.2:** Intro to Tooling

**Tooling** takes these techniques a step further by integrating external data sources, computational tools, and dynamic systems into the generation process. The key idea is that an LLM can be enhanced by leveraging additional resources outside of its own internal parameters to further adapt, retrieve, learn, and impact the environment on an orchestration level. 

### **Fundamentals of Tooling:**

For an LLM to interact with an external environment, **one** of the following must happen:
- **Observing System:** Its context can be enriched with some programmatic query which changes based on application state. 
    - **[Optionally]** It will be able to change the application state after recieving its state-dependent context. 
- **Interacting System:** It must can a query to an environment that will honor its request and change the application state. 
    - **[Optionally]** It will recieve feedback from the environment after it makes a request.

**An observing system is trivial to conceptualize,** since the only requirement is some semantically-understandable context. Fill in a prompt with some variables like the filenames inside a directory and you now have an LLM component that can tell you about your files. Allow a user to pick which document is fed into the LLM, and it's reasoning with a user's choice. The difficulty here is not "how to construct the context programmatically," but "what to put in it," and this difficulty is merely the difficulty of selection. 

**In contrast, an interacting system property is a bit harder to enforce,** since our LLM outputs are naturally unstructured. In the previous section, our LLMs were interacting with other LLMs since both are text-in text-out, but mapping to a regular non-LLM utility might be a bit more challenging, right?


#### **Grammar Enforcement**

Luckily for us, LLM sampling utilities and orchestration tools have worked hard to bridge this divide and allow us to call tools directly using an innovation called **grammar enforcement**!

Recalling all of our pipeline discussions from the earlier sections, algorithm input schemas are naturally defined as key-value-pair dictionaries:

```python
{
    "arg1": value1,
    "arg2": value2,
    ...
}
```

From an LLM perspective, assuming that the variable names and values are human-interpretable, the meaning can naturally be inferred and you could feed this in as context with no issues. Generating such a schema, however, is a bit less easy. Developers figured out early on that while you can "ask" an LLM for "valid python code" or "valid JSON with only the correct keys", such tactics usually required post-processing and failures would occur with some regularity. 

Nowadays, many models support **"schema"** inputs which specify a required format for the output. Consider the following tool instantiation, which creates an object that naturally works with this interface:

In [ ]:
from langchain.tools import tool

@tool
def add(
    explanation_of_what_the_user_wants: str, ## Optional. Gives some food for thought.
    a: float, 
    b: float
) -> int:
    """Adds a and b. Requires both arguments. Can be repurposed for subtraction"""
    return a + b

print(add.name)
print(add.description)
print(add.args)

When a serving stack receives a JSON schema, it can constrain decoding so the result has the
required keys and value types. That constraint guarantees structure, not meaning: field descriptions
and task instructions still tell the model what values belong in the schema.

`ChatNVIDIA.bind_tools` packages each tool description and schema for the model and returns selected
calls in `AIMessage.tool_calls`. For direct structured generation, current vLLM servers accept the
schema through `structured_outputs={"json": schema}`. We will inspect the generated arguments before
executing the tool.


In [ ]:
add.input_schema.model_json_schema()

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_nvidia import ChatNVIDIA
from random import random
import json

## In case you restart the notebook, you can start here
model_path = "http://127.0.0.1:9004/v1"
%env NVIDIA_BASE_URL=$model_path
%env NVIDIA_DEFAULT_MODE=open

model_name, using_vllm = "nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16", True  
# model_name, using_vllm = "nvidia/nemotron-3.5-lightning-30b-a3b", False  
## The hosted text alternative uses the same OpenAI-compatible interface.

llm = ChatNVIDIA(
    model=model_name, 
    max_completion_tokens=5000, 
    temperature=0
)

if using_vllm:
    ## VLLM offers a more customizable API than OpenAI's API requires
    add_tool = llm.bind(structured_outputs={"json": add.input_schema.model_json_schema()}) | StrOutputParser() | json.loads
else: 
    ## OpenAI API offers less flexibility, but has a simpler interface for what it does allow
    add_tool = llm.with_structured_output(add.input_schema) | dict

## The json schema can be processed into an instruction prompt for the endpoint to listen to
add_prompt = PydanticOutputParser(pydantic_object=add.input_schema).get_format_instructions()

## The following predicts the arguments to add, computes the "result", and passes the "args" through  
add_pipe = add_tool.bind(temperature=0) | {"args": RunnablePassthrough(), "result": add}

for i in range(10):

    a, b = random() * 10e10, random() * 10e10 
    # a, b = round(a), round(b)
    op_str, op_fn = "Add", lambda x,y: x+y
    # op_str, op_fn = "Sub", lambda x,y: x-y  ## You'll note that the models may have trouble subtracting with Add
    
    tool_output = add_pipe.invoke([
        ("system", add_prompt),
        # ("system", f"Assume schema of {add.args}"),  ## Lighter reinforcement.
        ("user", f"{op_str} the values of {a} and {b}."),
    ])

    tool_a = tool_output.get("args").get("a")
    tool_b = tool_output.get("args").get("b")
    tool_result = tool_output.get("result")
    
    print("[PASSED]" if (op_fn(a, b)) == tool_result else "[FAILED]", end=" ")
    print(f"{tool_a} {op_str} {tool_b} = {tool_result} vs {op_fn(a, b)}" )
    
    if "explanation_of_what_the_user_wants" in tool_output.get("args"):
        print("\tThoughts:", tool_output.get("args").get("explanation_of_what_the_user_wants"))

In [ ]:
llm._client.last_inputs

**Things To Try Out:**
- What happens if you don't provide a system message? Does it still work?
- What happens if you move the system messages back into the user messages? Does it work as well?
- What happens if you don't include the `explanation_of_what_the_user_wants` variable? Does it perform better or worse?

On the microscopic level, you've made an LLM do addition! This is... honestly not that big of a deal, right? Well, even on that narrow scale, it's probably better than asking the LLM to do it for you...

In [ ]:
for i in range(10):

    a, b = random() * 10e10, random() * 10e10 
    a, b = round(a), round(b)             ## Todo, try commenting out
    op_str, op_fn = "Add", lambda x,y: x+y
    # op_str, op_fn = "Sub", lambda x,y: x-y 
    
    tool_output = (llm | StrOutputParser()).invoke([
        ("user", f"{op_str} the values of {a} and {b}. Only return the final answer, not the arithmetic"),
    ])
    
    print("[PASSED]" if str(op_fn(a,b)) in tool_output else "[FAILED]", end=" ")
    print(f"{a} {op_str} {b} = {tool_output} vs {op_fn(a,b)}" )

<br>

On a macroscopic level though, we can extend structured output generation to allow our LLMs to adhere to almost any predefined output schema. This capability is crucial as we move toward treating LLMs as core components of a larger software pipeline.

<hr>
<br>

## **Part 7.3:** Multi-Tool Agentic Systems

We said earlier that interacting with an environment requires a system that can either observe and reason about a dynamic environment or impact and react to it directly. It turns out that a system that can do both is known to be **agentic**. 

> More generally, a system is ***agentic*** if it can observe, think about, react to, and act upon an environment with a personal directive. 
>
> Put another way, a system is ***agentic*** if it can pick a tool or action based on its current state, use it to impact something, and understand how its decision progresses it towards a goal.

**At a bare minimum, an agent generally needs only one component:** 
- A routing mechanism to pick a pathway.

**Further than that, many agents also incorporate:**
- A prediction of the arguments for the chosen pathway, if necessary.
- A buffer to accumulate memory, if necessary.
- An overarching or selectively-applied directive, if necessary.

<div><img src="imgs/simple-agent.png" width="600"/></div>

<br>

By that definition, the following example is technically a basic agentic loop that minimally-satisfies as an agent:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from random import random

sys_msg = (
    "Please help the user. After every response, output '[stop]`"
    " if the conversation should end and [pass] otherwise."
)

prompt = ChatPromptTemplate.from_messages([("system", sys_msg), ("placeholder", "{messages}")])
chain = prompt | llm | StrOutputParser()

state = {"messages": []}
agent_msg = ""

## NOTE: TODOs aleady initialized to streamline student workflows
while True:
    try: 
        ## TODO: Stop the event loop if the user asks to [stop]
        
        ## Update the messages appropriately
        human_msg = input("\n[Human]:")
        state["messages"] += [("human", human_msg)]

        ## Initiate an agent buffer to accumulate agent response
        agent_msg = ""
        print("\n[Agent]: ", end="")
        ## TODO: Stream the LLM's response from chain to output and accumulate it

        ## Update the messages list appropriately
        state["messages"] += [("ai", agent_msg)]

    except KeyboardInterrupt:
        print("KeyboardInterrupt")
        break

<details>
<summary><b>Solution</b></summary>

```python
## TODO: Stop the event loop if the user asks to [stop]
if "[stop]" in agent_msg: break

## TODO: Stream the LLM's response directly to output and accumulate it
for token in chain.stream(state):
    agent_msg += token
    print(token, end="")
```

</details>

<br>

Under our agentic abstraction, the loop above uses a simple *"does the output contain `[stop]`"* heuristic to decide whether to continue the conversation or not. Is this trivial? Yes, and it's technically just a routing mechanism `"[stop]" in agent_msg` and a subsequent tool call (`break`), but its logical extension is surprisingly powerful! 

- **[Multi-Tool]** What if we had more tools than just `pass` and `break`?
- **[State Management]** What if our tool choice (and the tool's argument choice) changes the behavior of our system?
- **[Retrieval]** What if our tool enriched out context with relevant information?
- **[Agentic Multimodality]** What if our tools allowed the agent to output other modalities (images, audio, video, etc) but only when necessary?

### **A Simple Multi-Tool Agent**

Of the classes above, **multi-tool agentics** is the most encompassing since almost all of the examples *could* fall under it. To explore this concept, let's create a calculator agent with the following tools:

In [ ]:
from langchain_core.tools import tool
from typing import Literal
import logging 

class tools:
    
    @tool
    def calculate_sum(a: float, b: float) -> float:
        """Adds a and b. Requires both arguments."""
        return float(a) + float(b)
    
    @tool
    def calculate_difference(a: float, b: float) -> float:
        """Subtracts a and b. Requires both arguments."""
        return float(a) - float(b)
    
    @tool
    def calculate_product(a: float, b: float) -> float:
        """Multiplies a and b. Requires both arguments."""
        return float(a) * float(b)
    
    @tool
    def calculate_quotient(a: float, b: float) -> float:
        """Divides a by b. Requires both arguments."""
        return float(a) / float(b)
    
    @tool
    def calculate_power(a: float, b: float) -> float:
        """Raises a to the power of b. Requires both arguments."""
        return float(a) ** float(b)
    
    @tool
    def no_tool() -> str:
        """Null tool; says no tool should be used"""
        return "No Tool Selected"

toolset = [v for k,v in tools.__dict__.items() if not k.startswith("_")]
toolset

If we wanted to, we **could** create a routing mechanism to first predict which tool to use and then call the tool. However, this process is so common and useful that many systems support it on the server-side in much the same way as structured output:


In [ ]:
from langchain_core.runnables import Runnable, RunnableAssign, RunnablePassthrough, RunnableLambda
from langgraph.prebuilt import ToolNode

# model_name, using_vllm = "nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16", True  
model_name, using_vllm = "nvidia/nemotron-3.5-lightning-30b-a3b", False  
## Use the reasoning model for this explicit agentic tool-use path.

llm = ChatNVIDIA(
    model=model_name, 
    max_completion_tokens=5000, 
    temperature=1,
    top_p=0.95,
)

## Create a client-side resolved which executes the tool picked by the server.
tool_node = ToolNode(toolset)

math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You're a math bot! Help the user as much as possible. Use the tools!" + str(tool_node.tools_by_name).replace("{", "{{").replace("}", "}}")),
    ("placeholder", "{messages}"),
])

tool_pipe = (
    math_prompt 
    ## Bind the tools to the connector, effectively feeding in the possible schemas on every invocation.
    | llm.bind_tools(
        toolset,
        chat_template_kwargs={"enable_thinking": True},
        reasoning_budget=2048,
    )
    # | {"messages": lambda x: [x]} | tool_node | RunnableLambda(lambda x: x.get("messages"))
)
tool_pipe.invoke({"messages": [("user", "What's 56766*30432?")]})

In [ ]:
## NOTE: The following are challenge examples, some of which may produce illegal values.
## Consider how you might actually support either the predicted argument support 
## OR improved decomposition strategies.

## TODO: Try both models. You may see some surprising results.

question = "What's 333333*555555?"
print(f"\nSimple {question = }")
print(tool_pipe.invoke({"messages": [("user", question)]}))

question = "What's 333333*555555 and 444444+222222?"
print(f"\nDouble {question = }")
print(tool_pipe.invoke({"messages": [("user", question)]}))

question = "What's 555555 times (444444 plus 222222)?"
print(f"\nComplex {question = }")
print(tool_pipe.invoke({"messages": [("user", question)]}))

question = "Introduce yourself in 10 words or less!"
print(f"\nRandom {question = }")
print(tool_pipe.invoke({"messages": [("user", question)]}))

In [ ]:
# llm._client.last_inputs

**As you can see from this demo:**
- `AIMessage.content` contains the visible answer, when one is returned.
- `AIMessage.tool_calls` contains structured calls selected by the model.
- `llm._client.last_inputs` shows the request fields sent by this client.

Tool choice, parallel calls, and schema handling vary by model and serving configuration. You can select a specific tool with `tool_choice`, or let the model choose whether to call a tool.


### Conversational Tool-Calling

Using some more advanced LLM orchestration paradigms, we can improve on our default tool-calling implementation by making a service that can do **both conversation and tool-calling in what seems like a single generation**. This is a more weedy topic and we will not go into much detail, but there several important reasons to bring this up:
- The intuitions associated with server-siding agentic capabilities underpins much of the more advanced custom abilities like tool-aware endpoints, server-side reflection/chain-of-thought, and endpoint-specific knowledge bases. 
- Here we construct a tag-based wrapper to see how conversation and tool calls can share a response. In Notebook 7.6, we will use the native tool-calling interface with LangGraph.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.prompts import ChatPromptTemplate
from operator import itemgetter
from copy import deepcopy
import re
legacy_llm = ChatNVIDIA(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    base_url="http://llm_client:9000/v1",
    temperature=0,
)

tool_instruction = (
    "You have access to the tools listed in the toolbank. Invoke one tool at a time using "
    "function tags around a JSON object containing name and parameters."
    " Select a tool for calculations or external information that should not be guessed."
    "\n\n<toolbank>{toolbank}</toolbank>\n"
    "Example (with a hypothetical tool):"
    "\nI will calculate the expression."
    "\n<function>{{\"name\":\"calculator\",\"parameters\":{{\"expression\":\"5 + 10\"}}}}</function>"
)

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a math bot! Help the user as much as possible by using the tools to answer their questions."
        " Give a brief action summary when useful, and use the tools for calculations."
        " You must abide by requirements, and should perform operations in order."
        " Recursive calls are not supported, so make sure to perform one command at a time."
        f"\n{tool_instruction}"
    )),
    ("placeholder", "{messages}"),
])

def parse_function_call(input_str):
    """Parses a function call string and extracts the function name and arguments."""
    # Matches the content between the <function> tags
    args_match = re.search(r'<function>(.+?)</function>', input_str, re.DOTALL)
    args = {}
    if args_match:
        # Get the raw JSON string payload
        json_payload_str = args_match.group(1).strip()
        try:
            # The payload is already a JSON object string. Load it directly.
            payload, end = json.JSONDecoder().raw_decode(json_payload_str)
            if not isinstance(payload, dict):
                raise ValueError("Tool-call JSON must be an object.")
            trailing = json_payload_str[end:].strip()
            if trailing not in {"", "</function"}:
                raise ValueError(f"Unexpected text after tool-call JSON: {trailing}")
            # Extract name and parameters (arguments) from the loaded JSON structure
            function_name = payload.get("name")
            tool_args = payload.get("parameters", {})
            if not isinstance(function_name, str) or not function_name or not isinstance(tool_args, dict):
                raise ValueError("Tool calls require a name and a parameters object.")
            # Return the unified dictionary format
            return {"name": function_name, "args": tool_args}
        except json.JSONDecodeError as e:
            # Re-raise with a clear error message
            raise ValueError(f"Invalid JSON in function arguments: {json_payload_str}. Error: {e}")            
    raise ValueError("Expected <function> with a JSON name and parameters object.")

def get_tool_result(tool_call, tools):
    """Find and run the tool based on the agent's function call."""
    tool_name = tool_call.get("name") or tool_call.get("function")
    tool_args = tool_call.get("args") or tool_call.get("parameters", {})
    tool = next((t for t in tools if t.name == tool_name), None)
    return tool.invoke(tool_args) if tool else f"Tool '{tool_name}' not found."

def agent_stream(toolset, state):
    working_state = deepcopy(state)
    
    ## Get a string version of the toolbank to add to the prompt to tell the systems about the tools
    toolbank = [str(convert_to_openai_tool(tool)).replace("{", "{{").replace("}", "}}") for tool in toolset]
    # Assuming 'RunnablePassthrough' is defined
    add_toolbank = RunnablePassthrough.assign(toolbank=lambda state: toolbank)
    # print(f"{toolbank = }\n")
    
    ## Create a stream-able conversation pipeline that is encouraged to chat conversationally
    # Assuming 'llm' and 'StrOutputParser' are defined
    chat_pipe = add_toolbank | agent_prompt | legacy_llm.bind(stop=["</function>"]) | StrOutputParser()
    chat_message = ""
    for token in chat_pipe.stream(working_state):
        yield token
        chat_message += token
    
    ## Early termination condition. No tool was called, so get out
    # FIXED: Remove the incorrect 'else: yield "</function>"' which yielded the closing tag prematurely
    if "<function" not in chat_message: return
    
    # Complete an interrupted closing tag, or append one when the model omitted it.
    if chat_message.rstrip().endswith("</function"):
        yield ">"
        chat_message += ">"
    else:
        yield "</function>"
        chat_message += "</function>"
    yield parse_function_call(chat_message)
 
input_state = {"messages": [("user", "What's (56766*30432+3043)/99?")]}

for chunk in agent_stream(toolset, input_state):
    if isinstance(chunk, str):
        print(chunk, end="") ## Keep outputting from conversational stream until a dict is found
    elif isinstance(chunk, dict):
        print(f"\nReceived Structured Output: {chunk}")
        tool_run = get_tool_result(chunk, toolset)
        print(f"\nResult: {tool_run}")

<br>

#### **Tangent:** The Inefficiency and Flexibility Balance

This setup is quite useful but relatively inefficient when compared to a first-class support option, in which the model first generates the conversational response, and then shifts gears into structured output decoding when the `<function>` tag is found. **This is a bit harder to implement, requires direct access to the token sampling layer of abstraction, and is not usually supported by the model developers since the model is not explicitly trained to do this.** 

With that being said, this **multi-pipeline orchestration**:
- with some values being streamed, some values being invoked...
- some things having one instruction, and others having completely different ones...
- some components using one model, others using completely different models, and some being algorthmically-defined or even just human interfaces...
- and the entire process cascading from an initial invokation which includes process enqueueing, branching, and termination...

**... is highly emblematic of the agentic software engineering paradigm of treating things as modules, allowing the modules to perform the tasks they're able to, and treating them as atomic primitives to organize higher-order tasks when the modules alone are insufficient**. Note that this itself mimics regular old user-centric software engineering, but that's a topic for another class.

<hr>
<br>

## **[Exercise] Part 7.4:** Enabling The Agentic Loop

Now that we have a conversational tool-calling endpoint, we should now be all set up to create **something** resembling an agentic loop. 
- In the previous notebook, we set up the **simple chat loop** that allowed us to accumulate a chat history.
- In this notebook, we will be setting up an **multi-step agentic loop** which will allow our system to call multiple tools before getting back to the user with a final answer.

<div><img src="imgs/basic-react.png" width="600"/></div>

We will elect to implement the [**ReAct (Reason+Act)** loop](https://arxiv.org/abs/2210.03629) which makes a simple assumption: 

**Keep calling for and observing the results of tool calls in an interleaving manner until you have a final answer.**

Sometimes this is as explicit as actually making "Final Answer" and "Ask The User" tools, and other times it can be implied by returning control to the user when no tool is called. We will pick the later, since our endpoint now supports conversation and tool-calling, all within a single request. Below, please test out the pipeline below and validate if it's working sufficiently. Consider ways you might improve the pipeline if it seems unreliable.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
import uuid

state = {"messages": []}

while True:
    print("\nCurrent Messages:", state["messages"])

    try:
        last_message = state["messages"][-1] if state["messages"] else None
        if last_message is None or isinstance(last_message, AIMessage):
            user_input = input("\n[Human]: ")
            if user_input.strip().lower() in ["exit", "quit"]:
                print("Exiting...")
                break
            state["messages"].append(("human", user_input))

        print("\n[Agent]: ", end="")
        response = ""
        tool_call = None
        tool_result = None
        for chunk in agent_stream(toolset, state):
            if isinstance(chunk, str):
                response += chunk
                print(chunk, end="")
            elif isinstance(chunk, dict):
                tool_call = chunk
                tool_result = get_tool_result(chunk, toolset)

        if tool_call:
            call_id = f"call_{uuid.uuid4().hex[:8]}"
            state["messages"].append(AIMessage(
                content=response,
                tool_calls=[{
                    "name": tool_call["name"],
                    "args": tool_call["args"],
                    "id": call_id,
                    "type": "tool_call",
                }],
            ))
            state["messages"].append(ToolMessage(
                content=str(tool_result),
                tool_call_id=call_id,
            ))
        elif response:
            state["messages"].append(AIMessage(content=response))

    except KeyboardInterrupt:
        print("\nExiting via KeyboardInterrupt.")
        break


**Potential Question:** 
- What's (56766*30432+3043)/99?

**Answer Progression:**
- Step 1: 1727502912
- Step 2: 1727505955
- Step 3: 17449555.101010103

**Challenge Question:**
- Compute the fibonacci sequence up to 25.
- Now use tools to compute the digits 26 - 30.
- How much bigger is the 30th number vs the 25th?

<hr>
<br>

# <font color="#76b900">**Wrapping Up**</font>

In this section, we introduced stateful LLM systems and the logic that underpins more complex and controllable (or even attempted self-controlling) formulations! This is really only the beginning of the exciting things you can do with LLMs, but hopefully you've enjoyed this introduction! 

Going from the limited formulations of task-specific encoders to the set of powerful generative and self-guiding models, we hope you can appreciate that every model we've discussed serves some place in the stack! Whether it's a key backbone component, a cheap complementary mechanism, a source of inspiration for further development, or a stepping stone to advancements in the field, try to keep these tools in your arsenal going forward and use the ones that work best for your settings!

**Before the assessment, Notebook 7.5 prepares the image pipeline and Notebook 7.6 rebuilds the agent loop with LangGraph.**

In [ ]:
# ## Please Run When You're Done!
# import IPython
# app = IPython.Application.instance()
# app.kernel.do_shutdown(True)

<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>